# PULSAR MCT + PBMC VAE Token Injection

Tests whether adding a per-sample pseudobulk VAE embedding (z_bio) as an additional token to PULSAR's Multicellular Transformer improves lupus classification.

**Runtime**: GPU required (T4 or A100). Set in `Runtime → Change runtime type`.

**Data needed**: upload 4 files from `vae_health/analysis/results/q62_pbmc_vae/`:
- `uce_batches_256.npy` (326 MB) — 261 donors × 256 cells × 1280-d UCE
- `z_bio.npy` (16 KB) — 261 × 16 VAE embeddings
- `donor_meta.csv` (10 KB) — donor labels
- `pbmc_vae.pt` (14 MB) — VAE checkpoint (optional, for re-encoding)

In [ ]:
# Install — pin transformers to version compatible with torch on Colab
!pip install -q "transformers==4.41.0" huggingface_hub safetensors
!git clone -q --depth 1 https://github.com/snap-stanford/PULSAR /tmp/pulsar_repo 2>/dev/null || echo "already cloned"
import sys; sys.path.insert(0, "/tmp/pulsar_repo/src")
print("Setup done")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"torch: {torch.__version__}")

## 2. Load Data
Upload the 4 files from `q62_pbmc_vae/` when prompted.

In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

# ── Option A: copy from Google Drive ─────────────────────────────────────────
# Put the 4 files in a folder called q62_pbmc_vae inside your Google Drive root
DRIVE_DIR = '/content/drive/MyDrive/q62_pbmc_vae'
REQUIRED  = ['uce_batches_256.npy', 'z_bio.npy', 'donor_meta.csv', 'pbmc_vae.pt']

if os.path.isdir(DRIVE_DIR):
    for fn in REQUIRED:
        src = os.path.join(DRIVE_DIR, fn)
        if os.path.exists(src):
            shutil.copy(src, f'/content/{fn}')
            print(f'  copied {fn}  ({os.path.getsize("/content/"+fn)/1e6:.1f} MB)')
        else:
            print(f'  MISSING in Drive: {fn}')
    print("Done — all files in /content/")
else:
    # ── Option B: manual upload fallback ─────────────────────────────────────
    print(f'Drive folder not found: {DRIVE_DIR}')
    print('Upload files manually:')
    from google.colab import files as _f
    up = _f.upload()
    for fn, data in up.items():
        open(fn, 'wb').write(data)
        print(f'  saved {fn}')


In [ ]:
# Load artefacts
uce_batches = np.load("uce_batches_256.npy")  # (261, 256, 1280)
z_bio       = np.load("z_bio.npy")            # (261, 16)
meta_df     = pd.read_csv("donor_meta.csv")

y = meta_df["label"].values.astype(np.int64)  # 1=lupus, 0=normal
print(f"UCE batches:  {uce_batches.shape}")
print(f"z_bio:        {z_bio.shape}")
print(f"Donors: {len(y)}  |  lupus: {y.sum()}  normal: {(y==0).sum()}")

## 3. PULSAR Model

Load pretrained `PULSAR-pbmc` (87.4M params). We copy the config and instantiate a standalone PyTorch model, then load HuggingFace weights.

In [ ]:
# Load pretrained PULSAR-pbmc (87.4M params) from HuggingFace
pulsar = PULSAR.from_pretrained("KuanP/PULSAR-pbmc").to(DEVICE)

n_params = sum(p.numel() for p in pulsar.parameters())
print(f"PULSAR loaded: {n_params/1e6:.1f}M params")
print(f"  hidden_size={pulsar.config.hidden_size}, encoder_layers={pulsar.config.encoder_num_hidden_layers}")
pulsar.eval()
print("  mode: eval")


## 4. Baseline: Frozen PULSAR + Linear Probe

Mean-pool UCE per donor → PULSAR CLS token → LogisticRegression  
This replicates the Q61 result with the **actual** 512-d PULSAR CLS embedding (vs. the 1280-d UCE mean-pool used before).

In [ ]:
@torch.no_grad()
def get_pulsar_cls(uce_batch_np, pulsar_model, batch_size=16):
    """Run PULSAR encoder on (N, 256, 1280) UCE batches → (N, 768) CLS tokens."""
    pulsar_model.eval()
    N = uce_batch_np.shape[0]
    cls_list = []
    for i in range(0, N, batch_size):
        x = torch.from_numpy(uce_batch_np[i:i+batch_size]).to(DEVICE)
        enc_out = pulsar_model.encode(x)          # (B, 257, 768)
        cls_list.append(enc_out[0][:, 0, :].cpu().numpy())
    return np.vstack(cls_list)

print("Extracting PULSAR CLS embeddings (frozen)...")
cls_emb = get_pulsar_cls(uce_batches, pulsar)
print(f"CLS embeddings: {cls_emb.shape}")

In [ ]:
def cv_classify(X, y, label, C=0.1, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_sc = StandardScaler().fit_transform(X)
    accs, f1s = [], []
    for tr, te in kf.split(X_sc, y):
        clf = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=42)
        clf.fit(X_sc[tr], y[tr])
        yp = clf.predict(X_sc[te])
        accs.append(accuracy_score(y[te], yp))
        f1s.append(f1_score(y[te], yp, average="macro"))
    acc_m, acc_s = np.mean(accs), np.std(accs)
    f1_m,  f1_s  = np.mean(f1s),  np.std(f1s)
    print(f"  {label:<45}  acc={acc_m:.3f}\u00b1{acc_s:.3f}  f1={f1_m:.3f}\u00b1{f1_s:.3f}")
    return {"label": label, "acc": acc_m, "f1": f1_m, "acc_std": acc_s, "f1_std": f1_s}

results = []
print("5-fold CV logistic regression probes:")
results.append(cv_classify(cls_emb, y,  "A. PULSAR CLS 768-d (baseline)"))
results.append(cv_classify(z_bio,   y,  "B. VAE z_bio 16-d alone"))
results.append(cv_classify(np.hstack([cls_emb, z_bio]), y, "C. PULSAR CLS + z_bio (concat probe)"))


## 5. PULSAR + VAE Token Injection

Modify the MCT to accept a **pseudobulk token**: project z_bio (16-d → 768-d) and prepend it between the CLS token and the cell embeddings.

```
[CLS]  [VAE_token]  [cell_0]  [cell_1]  ...  [cell_255]
  ↑         ↑
learned  projected from z_bio (16-d)
```
The self-attention can then weight the pseudobulk summary against individual cells.

In [ ]:
        cls_tok = self.pulsar.cls_embedding.unsqueeze(0).expand(B, 1, -1).clone()

In [ ]:
    # deepcopy on CPU to avoid GPU memory fragmentation across folds
    pulsar_copy = copy.deepcopy(pulsar_pretrained.cpu()).to(DEVICE)
    pulsar_pretrained = pulsar_pretrained.to(DEVICE)  # restore original to GPU


## 6. Run Fine-tuning Experiments

Three strategies in order of compute cost. Run them top to bottom; if frozen already reaches plateau, partial/full may not be needed.

In [ ]:
finetune_results = []

print("\n=== Strategy D: frozen PULSAR + VAE token + classifier ===")
print("  (Only projector + head are trained; ~1M params)")
r = cv_finetune(pulsar, uce_batches, z_bio, y,
                strategy="frozen", label="D. PULSAR+VAE frozen (20 ep)",
                epochs=20, lr=2e-4)
finetune_results.append(r)

In [ ]:
print("\n=== Strategy E: partial unfreeze (top-4 MCT layers) + VAE token ===")
r = cv_finetune(pulsar, uce_batches, z_bio, y,
                strategy="partial", label="E. PULSAR+VAE partial (20 ep)",
                epochs=20, lr=1e-4)
finetune_results.append(r)

In [ ]:
print("\n=== Strategy F: full fine-tune PULSAR + VAE token ===")
print("  (PULSAR at LR/10, projector+head at full LR)")
r = cv_finetune(pulsar, uce_batches, z_bio, y,
                strategy="full", label="F. PULSAR+VAE full FT (20 ep)",
                epochs=20, lr=5e-5)
finetune_results.append(r)

## 7. Ablation: Fine-tune PULSAR *without* VAE token

Same training budget, but the VAE token is replaced by a zero vector. This isolates the contribution of z_bio vs. just fine-tuning.

In [ ]:
    # deepcopy on CPU to avoid GPU memory fragmentation across folds
    pulsar_copy = copy.deepcopy(pulsar_pretrained.cpu()).to(DEVICE)
    pulsar_pretrained = pulsar_pretrained.to(DEVICE)  # restore original to GPU


In [ ]:
print("\n=== Ablation: frozen PULSAR, z_bio zeroed out ===")
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ab_accs, ab_f1s = [], []
for fold_i, (tr, te) in enumerate(kf.split(uce_batches, y)):
    acc, f1 = train_one_fold_ablated(pulsar, uce_batches, z_bio, y, tr, te, "frozen")
    ab_accs.append(acc); ab_f1s.append(f1)
ablated = {"label": "G. Ablation: PULSAR frozen, z=0 (frozen)",
           "acc": np.mean(ab_accs), "f1": np.mean(ab_f1s),
           "acc_std": np.std(ab_accs), "f1_std": np.std(ab_f1s)}
finetune_results.append(ablated)
print(f"  {ablated['label']}  acc={ablated['acc']:.3f}  f1={ablated['f1']:.3f}")

## 8. Results

Compare all probe + fine-tune results side by side.

In [ ]:
all_results = results + finetune_results  # probe + finetune

print("\n" + "="*70)
print("LUPUS CLASSIFICATION — 5-fold CV (261 donors)")
print("="*70)
print(f"{'Method':<47}  {'Acc':>7}  {'F1-macro':>10}")
print("-"*70)
for r in all_results:
    print(f"  {r['label']:<45}  {r['acc']:.3f}±{r['acc_std']:.3f}  {r['f1']:.3f}±{r['f1_std']:.3f}")

# Δ
baseline_f1 = results[0]["f1"]
best_vae_f1 = max(r["f1"] for r in finetune_results if "Ablation" not in r["label"])
print(f"\n  Δ F1 (best VAE strategy vs PULSAR baseline probe): {best_vae_f1 - baseline_f1:+.3f}")

In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
labels_all = [r["label"].split(". ", 1)[1] for r in all_results]
f1_vals  = [r["f1"]       for r in all_results]
f1_errs  = [r["f1_std"]   for r in all_results]
colors = (["steelblue"] * len(results) +
          ["orange" if "Ablation" not in r["label"] else "gray" for r in finetune_results])
x = range(len(all_results))
ax.bar(x, f1_vals, yerr=f1_errs, capsize=4, color=colors, alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(labels_all, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Macro-F1 (5-fold CV)"); ax.set_ylim(0.6, 1.0)
ax.set_title("PULSAR MCT + PBMC VAE Token — Lupus Classification")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="steelblue", label="linear probe"),
                   Patch(color="orange", label="fine-tune + VAE token"),
                   Patch(color="gray", label="ablation (z=0)")])
plt.tight_layout()
plt.savefig("pulsar_vae_results.png", dpi=150)
plt.show()

## 9. If VAE token doesn't help: alternative injection strategies

Run this section if the VAE token Δ F1 < 0.01.

In [ ]:
# Strategy H: CLS + z_bio concatenation (skip the extra token, just fuse at head)
class CLSPlusVAEHead(nn.Module):
    """Concat 768-d CLS + 16-d z_bio → wide MLP classifier.  No extra token."""
    def __init__(self, cls_dim=768, z_dim=16, num_labels=2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(cls_dim + z_dim, (cls_dim + z_dim) * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear((cls_dim + z_dim) * 2, num_labels),
        )
    def forward(self, cls, z, labels=None):
        logits = self.classifier(torch.cat([cls, z], dim=-1))
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(
                weight=torch.tensor([1.62, 1.0], device=cls.device)
            )(logits, labels)
        return loss, logits

# Pre-extract frozen CLS for speed
print("Pre-extracting frozen PULSAR CLS embeddings for all donors...")
cls_all = get_pulsar_cls(uce_batches, pulsar)  # (261, 768)
cls_t   = torch.from_numpy(cls_all.astype(np.float32))
z_t     = torch.from_numpy(z_bio.astype(np.float32))
y_t     = torch.from_numpy(y)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
h_accs, h_f1s = [], []
for tr, te in kf.split(cls_all, y):
    head = CLSPlusVAEHead().to(DEVICE)
    opt  = optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    ds   = TensorDataset(cls_t[tr], z_t[tr], y_t[tr])
    dl   = DataLoader(ds, batch_size=16, shuffle=True)
    for ep in range(50):
        head.train()
        for c, z_, lab in dl:
            c, z_, lab = c.to(DEVICE), z_.to(DEVICE), lab.to(DEVICE)
            opt.zero_grad()
            loss, _ = head(c, z_, lab)
            loss.backward(); opt.step()
    head.eval()
    with torch.no_grad():
        _, logits = head(cls_t[te].to(DEVICE), z_t[te].to(DEVICE))
    preds = logits.argmax(-1).cpu().numpy()
    h_accs.append(accuracy_score(y[te], preds))
    h_f1s.append(f1_score(y[te], preds, average="macro"))
print(f"  H. CLS + z_bio concat head:  acc={np.mean(h_accs):.3f}  f1={np.mean(h_f1s):.3f}")

In [ ]:
        cls_tok = self.pulsar.cls_embedding.unsqueeze(0).expand(B, 1, -1).clone()